## Exercise 01

Fetch the data from:
https://makeup-api.herokuapp.com/api/v1/products.json?brand=maybelline

Then:
1. Find the name of the product of the maximum price

In [1]:
import json
from urllib.request import urlopen


url = 'https://makeup-api.herokuapp.com/api/v1/products.json?brand=maybelline'


with urlopen(url) as response:
    products = json.load(response)


valid_products = []
for product in products:
    price = product.get('price')
    if price not in (None, ''):
        valid_products.append({
            'name': product.get('name'),
            'price': float(price)
        })


most_expensive_product = max(valid_products, key=lambda item: item['price'])

print('Product with maximum price:', most_expensive_product['name'])
print('Price:', most_expensive_product['price'])

Product with maximum price: Maybelline Dream Velvet Foundation
Price: 18.49


2. Find the name of the product of the minimum price

In [2]:
least_expensive_product = min(valid_products, key=lambda item: item['price'])

print('Product with minimum price:', least_expensive_product['name'])
print('Price:', least_expensive_product['price'])

Product with minimum price: Maybelline Color Show Nail Lacquer Jewels 
Price: 4.49


3. Find the average rating of all maybelline products (exclude any product that has no rating)

In [3]:
valid_ratings = []
for product in products:
    rating = product.get('rating')
    if rating not in (None, ''):
        valid_ratings.append(float(rating))

if valid_ratings:
    average_rating = sum(valid_ratings) / len(valid_ratings)
    print('Average rating:', round(average_rating, 2))
else:
    print('No products with ratings found.')

Average rating: 4.08


## Exercise 02

Visit https://www.usatoday.com/news/  then for each of the following categories that you see on the top of the homepage: (`sports`, `entertaiment`, `life`), 

<br/><img src="https://raw.githubusercontent.com/ha2285/tmp/main/1.png" width="60%" high="60%"><br/>


- **<u>Try</u>** to visit each of these links and extract some info about the articles found on the rightside of the category's main page.

<br/><img src="https://raw.githubusercontent.com/ha2285/tmp/main/2.png" width="60%" high="60%"><br/>

- For each article, you need to visit the article then extract:
`title`, `category`, `author`, `publishing date` and the `URL of this article`.

<br/><img src="https://raw.githubusercontent.com/ha2285/tmp/main/3.png" width="60%" high="60%"><br/>

- Save all these data into a text file called `articles.txt`. It should contain the data of each article in one line with comma's between in between, as follows:

`title`, `category`, `author`, `publishing date`, `URL`



In [4]:
import csv
import json
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

headers = {
    'User-Agent': 'Mozilla/5.0'
}

section_urls = {
    'sports': 'https://www.usatoday.com/sports/',
    'entertainment': 'https://www.usatoday.com/entertainment/',
    'life': 'https://www.usatoday.com/life/'
}

sidebar_selectors = (
    'a.gnt_m_th_a',
    'a[class*="gnt_m_th_a"]',
)


def get_soup(url):
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')


def normalize_story_url(section_url, href):
    full_url = urljoin(section_url, href.strip())
    parsed = urlparse(full_url)
    return f'{parsed.scheme}://{parsed.netloc}{parsed.path}'


def is_section_story(article_url, section_name):
    path_parts = [part for part in urlparse(article_url).path.lower().split('/') if part]
    return len(path_parts) >= 2 and path_parts[0] == 'story' and path_parts[1] == section_name


def collect_links(anchors, section_name, section_url, seen, limit=None):
    links = []

    for anchor in anchors:
        href = anchor.get('href')
        if not href:
            continue

        article_url = normalize_story_url(section_url, href)
        if article_url in seen or not is_section_story(article_url, section_name):
            continue

        seen.add(article_url)
        links.append(article_url)

        if limit is not None and len(links) >= limit:
            break

    return links


def get_meta_content(soup, attrs):
    tag = soup.find('meta', attrs=attrs)
    return tag.get('content', '').strip() if tag and tag.get('content') else ''


def get_article_json_ld(soup):
    for tag in soup.find_all('script', type='application/ld+json'):
        text = tag.get_text(strip=True)
        if not text:
            continue

        try:
            data = json.loads(text)
        except Exception:
            continue

        candidates = data if isinstance(data, list) else [data]
        stack = list(candidates)

        while stack:
            item = stack.pop(0)
            if not isinstance(item, dict):
                continue

            if item.get('@type') in ('NewsArticle', 'Article'):
                return item

            graph = item.get('@graph')
            if isinstance(graph, list):
                stack.extend(graph)

    return {}


def extract_article_links(section_name, section_url, limit=10):
    soup = get_soup(section_url)
    seen = set()
    links = []

    # Prefer the right-side story stack when the page exposes it.
    for selector in sidebar_selectors:
        sidebar_links = collect_links(
            soup.select(selector),
            section_name,
            section_url,
            seen,
        )
        if sidebar_links:
            links.extend(sidebar_links)
            break

    if not links:
        links.extend(
            collect_links(
                soup.find_all('a', href=True),
                section_name,
                section_url,
                seen,
                limit=limit,
            )
        )

    return links[:limit]


def extract_article_details(article_url, fallback_category):
    soup = get_soup(article_url)
    article_data = get_article_json_ld(soup)

    title = article_data.get('headline') or get_meta_content(soup, {'property': 'og:title'})
    if not title:
        title_tag = soup.find('h1')
        title = title_tag.get_text(strip=True) if title_tag else 'Unknown'

    category = article_data.get('articleSection') or get_meta_content(soup, {'property': 'article:section'}) or fallback_category

    author_data = article_data.get('author')
    if isinstance(author_data, dict):
        author = author_data.get('name', 'Unknown')
    elif isinstance(author_data, list) and author_data:
        first_author = author_data[0]
        author = first_author.get('name', 'Unknown') if isinstance(first_author, dict) else str(first_author)
    else:
        author = get_meta_content(soup, {'name': 'author'}) or 'Unknown'

    publishing_date = article_data.get('datePublished') or get_meta_content(soup, {'property': 'article:published_time'})
    if not publishing_date:
        time_tag = soup.find('time')
        publishing_date = time_tag.get('datetime', '').strip() if time_tag else ''

    return {
        'title': title,
        'category': category,
        'author': author,
        'publishing date': publishing_date,
        'URL': article_url
    }


articles = []
visited_urls = set()

for section_name, section_url in section_urls.items():
    try:
        article_links = extract_article_links(section_name, section_url)
    except Exception as error:
        print(f'Could not fetch {section_name}: {error}')
        continue

    for article_url in article_links:
        if article_url in visited_urls:
            continue

        visited_urls.add(article_url)

        try:
            articles.append(extract_article_details(article_url, section_name))
        except Exception as error:
            print(f'Could not parse {article_url}: {error}')

with open('articles.txt', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    for article in articles:
        writer.writerow([
            article['title'],
            article['category'],
            article['author'],
            article['publishing date'],
            article['URL'],
        ])

print(f'Saved {len(articles)} articles to articles.txt')


Saved 18 articles to articles.txt


- Write code to read the data from `articles.txt` as DataFrame, then find the following:
    - The number of articles per category
    - The latest article by each category
    - The author with the most published articles

In [5]:
import pandas as pd

column_names = ['title', 'category', 'author', 'publishing date', 'URL']

try:
    articles_df = pd.read_csv('articles.txt', names=column_names)
except FileNotFoundError:
    print('articles.txt was not found. Run the scraper cell first.')
else:
    if not articles_df.empty and articles_df.iloc[0].tolist() == column_names:
        articles_df = articles_df.iloc[1:].reset_index(drop=True)

    if articles_df.empty:
        print('articles.txt is empty. Run the scraper cell first and make sure the website is reachable.')
    else:
        articles_df['publishing date'] = pd.to_datetime(articles_df['publishing date'], errors='coerce')

        print('Number of articles per category:')
        print(articles_df['category'].value_counts())
        print()

        latest_articles = (
            articles_df.dropna(subset=['publishing date'])
            .sort_values('publishing date')
            .groupby('category', as_index=False)
            .tail(1)[['category', 'title', 'publishing date', 'URL']]
            .sort_values('category')
        )

        print('Latest article by each category:')
        print(latest_articles.to_string(index=False))
        print()

        known_authors = articles_df[articles_df['author'].fillna('').str.strip().ne('')]
        known_authors = known_authors[known_authors['author'].str.lower() != 'unknown']

        if known_authors.empty:
            print('Author with the most published articles: no author information available')
        else:
            top_author = known_authors['author'].value_counts().idxmax()
            top_count = known_authors['author'].value_counts().max()
            print(f'Author with the most published articles: {top_author} ({top_count} articles)')


Number of articles per category:
category
entertainment    10
sports            4
life              4
Name: count, dtype: int64

Latest article by each category:
     category                                                                                   title           publishing date                                                                                                                                                             URL
entertainment  Brandy says Boyz II Men romance started at 16, reveals final call with Whitney Houston 2026-03-31 11:02:10+00:00                                                     https://www.usatoday.com/story/entertainment/books/2026/03/31/brandy-memoir-phases-revelations/89386752007/
         life                                What exactly are GLP-1 medications and how do they work? 2026-03-30 18:59:47+00:00                                                                                  https://www.usatoday.com/story/life/health-wellness/w